# Beyond - Muon

This notebook implements Newton–Schulz orthogonalization and watches it flatten a spectrum, compares AdamW's and Muon's update geometry on a synthetic momentum matrix, races the two optimizers — each at its own best learning rate — on a char-level TinyShakespeare transformer, violates the matrix/embedding partition on purpose, and totals the optimizer-state bill.

1. Read the lesson page (`docs/beyond/muon.md`).
2. Open this notebook with `./notebook.sh muon`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

In [ ]:
import torch
import matplotlib.pyplot as plt

from g2c.muon import Muon, zeropower_via_newtonschulz

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")

## Exercise 1 — Watch the spectrum flatten

Run a random matrix through 1, 2, 3, and 5 Newton–Schulz passes and plot the singular-value spectrum after each. Spectra are computed on CPU tensors (see the lesson's M-series notes).

In [ ]:
torch.manual_seed(0)
G = torch.randn(64, 32)

plt.figure(figsize=(7, 4))
plt.plot(torch.linalg.svdvals(G / G.norm()), label="normalized input")
for steps in (1, 2, 3, 5):
    s = torch.linalg.svdvals(zeropower_via_newtonschulz(G, steps=steps))
    plt.plot(s, label=f"after {steps} NS pass{'es' if steps > 1 else ''}")
plt.xlabel("singular value index (descending)")
plt.ylabel("singular value")
plt.title("Newton–Schulz pushes the spectrum toward 1")
plt.legend()
plt.show()

In [ ]:
"Question: Describe what each successive Newton-Schulz pass did to the small singular values versus the ones already near 1, and connect the shape of the change to the quintic f(s) = 3.4445s - 4.775s^3 + 2.0315s^5. Why must the input be normalized by its Frobenius norm before the first pass?"
"Answer: "

## Exercise 2 — AdamW's update versus Muon's, geometrically

Early in training, momentum is often dominated by a few directions. Build a nearly rank-1 momentum matrix and compare the update each optimizer would take. (The per-coordinate stand-in `M / (|M| + eps)` is the standard sign-like approximation of an Adam-family step once its running scales have adapted.)

In [ ]:
torch.manual_seed(1)
u, v = torch.randn(64, 1), torch.randn(1, 32)
M = u @ v + 0.05 * torch.randn(64, 32)   # near-rank-1 momentum + noise

updates = {
    "raw momentum": M,
    "per-coordinate (AdamW-like)": M / (M.abs() + 1e-8),
    "Muon (orthogonalized)": zeropower_via_newtonschulz(M),
}

plt.figure(figsize=(7, 4))
for name, upd in updates.items():
    s = torch.linalg.svdvals(upd)
    plt.plot(s / s.max(), label=name)
plt.xlabel("singular value index")
plt.ylabel("gain relative to largest")
plt.title("Where each update spends its step")
plt.legend()
plt.show()

In [ ]:
"Question: Compare the three spectra: where does the raw momentum concentrate its energy, what does the per-coordinate update change, and what does orthogonalization change? Which update moves the weight along the small/noise directions, and why might that help or hurt training?"
"Answer: "

## Exercise 3 — The race, run fairly

Train the same char-level TinyShakespeare transformer under AdamW and under the Muon hybrid, each at its own best learning rate from the Exercise 4 sweep. The partition function is the design decision: matrices that act as maps go to Muon; the embedding table, positional table, norm gains, biases, and the unembedding bias stay on AdamW.

In [ ]:
from pathlib import Path

SHAKESPEARE = Path("data/datasets/tinyshakespeare.txt")
if not SHAKESPEARE.exists():
    raise RuntimeError(
        "TinyShakespeare is missing - run ./setup.sh first."
    )

text = SHAKESPEARE.read_text()
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
ids = torch.tensor([stoi[c] for c in text], dtype=torch.long)
VOCAB = len(chars)

from g2c.pretraining import split_token_stream

train_ids, val_ids = split_token_stream(ids, 0.9)
print(f"vocab {VOCAB}  train {train_ids.numel():,} tokens  "
      f"val {val_ids.numel():,}")

In [ ]:
from g2c.pretraining import get_lm_batch, lm_cross_entropy
from g2c.training import AdamW
from g2c.transformer import TransformerLM

model_cfg = dict(vocab_size=VOCAB, embedding_dim=128, num_layers=4,
                 num_heads=4, max_seq_len=128, hidden_dim=512)


def partition_params(model):
    """Matrices-as-maps -> Muon; everything else -> AdamW."""
    excluded = {id(p) for p in [*model.token_embed.parameters(),
                                *model.pos_embed.parameters()]}
    excluded.add(id(model.head_bias))
    matrices = [p for p in model.parameters()
                if p.ndim == 2 and id(p) not in excluded]
    matrix_ids = {id(m) for m in matrices}
    others = [p for p in model.parameters() if id(p) not in matrix_ids]
    return matrices, others


def make_optimizer(model, kind, lr):
    if kind == "adamw":
        return AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    matrices, others = partition_params(model)
    return Muon(matrices, others, lr=lr, weight_decay=0.01,
                adamw_lr=3e-4)


def train_char_lm(kind, lr, *, steps=800, batch_size=32, context=128,
                  eval_every=100, seed=0, optimizer_factory=None):
    torch.manual_seed(seed)
    model = TransformerLM(**model_cfg).to(device)
    factory = optimizer_factory or make_optimizer
    opt = factory(model, kind, lr)
    gen = torch.Generator().manual_seed(seed)
    history = {"step": [], "val_loss": []}
    for step in range(1, steps + 1):
        x, y = get_lm_batch(train_ids, batch_size, context,
                            generator=gen)
        opt.zero_grad()
        loss = lm_cross_entropy(model(x.to(device)), y.to(device))
        loss.backward()
        opt.step()
        if step % eval_every == 0 or step == steps:
            with torch.no_grad():
                vx, vy = get_lm_batch(val_ids, 64, context,
                                      generator=gen)
                val = lm_cross_entropy(
                    model(vx.to(device)), vy.to(device)
                ).item()
            history["step"].append(step)
            history["val_loss"].append(val)
            print(f"  {kind} lr={lr:g} step {step}: val {val:.3f}")
    return model, history

In [ ]:
BEST = {"adamw": 3e-4, "muon": 0.02}   # revisit after Exercise 4

histories = {}
for kind in ("adamw", "muon"):
    print(f"--- {kind} @ lr {BEST[kind]:g}")
    _, histories[kind] = train_char_lm(kind, BEST[kind], seed=3)

plt.figure(figsize=(7, 4))
for kind, h in histories.items():
    plt.plot(h["step"], h["val_loss"], label=f"{kind} @ {BEST[kind]:g}")
plt.xlabel("step")
plt.ylabel("val loss")
plt.legend()
plt.title("AdamW vs Muon, best-of-sweep each")
plt.show()

## Exercise 4 — The learning-rate landscape

Sweep each optimizer across its own plausible range, then go back and set `BEST` in the race cell above. Short runs are enough to rank learning rates.

In [ ]:
SWEEPS = {"adamw": (1e-4, 3e-4, 1e-3),
          "muon": (0.005, 0.02, 0.08)}

finals = {}
for kind, lrs in SWEEPS.items():
    for lr in lrs:
        _, h = train_char_lm(kind, lr, steps=300, eval_every=300,
                             seed=7)
        finals[(kind, lr)] = h["val_loss"][-1]

print()
for (kind, lr), val in sorted(finals.items()):
    print(f"{kind:6s} lr={lr:<8g} final val {val:.3f}")

In [ ]:
"Question: Report the race: which optimizer won at its own best learning rate, by how much, and how far apart were the two best learning rates? Which optimizer degraded more gracefully when its learning rate was off by ~3x? If the result surprised you or looked like a tie, say so and name the control (another seed, longer runs) you would run next."
"Answer: "

## Exercise 5 — Violate the partition

Route every 2-D parameter through Muon — including the embedding table, the positional table, and (because the unembedding is tied) the output side too — and rerun the race configuration. The lesson's argument says embedding rows are unrelated per-token objects; see whether the damage is visible at this scale.

In [ ]:
def make_muon_everything(model, kind, lr):
    matrices = [p for p in model.parameters() if p.ndim == 2]
    others = [p for p in model.parameters() if p.ndim != 2]
    return Muon(matrices, others, lr=lr, weight_decay=0.01,
                adamw_lr=3e-4)


_, history_bad = train_char_lm("muon-everything", BEST["muon"],
                               seed=3,
                               optimizer_factory=make_muon_everything)

plt.figure(figsize=(7, 4))
plt.plot(histories["muon"]["step"], histories["muon"]["val_loss"],
         label="muon (proper partition)")
plt.plot(history_bad["step"], history_bad["val_loss"],
         label="muon-everything")
plt.xlabel("step")
plt.ylabel("val loss")
plt.legend()
plt.title("The partition, violated")
plt.show()

In [ ]:
"Question: What did routing the embedding and positional tables through Muon do to the curves, if anything? Explain the rows-are-unrelated-objects argument for why the partition rule exists, and - if the toy run hid the damage - why a 65-character vocabulary might mask what a 100k-token vocabulary would not."
"Answer: "

## Exercise 6 — Written: the state bill

In [ ]:
"Question: For the race model, compute optimizer-state memory (float32) under AdamW-everything versus the Muon hybrid: count parameters on each side of the partition, apply two buffers per AdamW parameter and one per Muon parameter, and give both totals and the saving. Connect the result to Module 13B's memory-tenant table."
"Answer: "

When complete, ask a coding agent to grade your notebook. Partial work is fine: the agent should grade answered questions and implemented sections, then skip blank prompts.